# Disque 100 - Auditoria, Limpeza, Normalização e Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from collections import defaultdict

## 1. Snapshot da base

In [ ]:
df_raw = df.copy()
print(df.shape)

## 2. Auditoria de completude

In [ ]:
completude = pd.DataFrame({
    "missing": df.isna().sum(),
    "percentual": round(df.isna().mean()*100,2)
}).sort_values("percentual", ascending=False)
completude

## 3. Normalização global de texto (remove acentos, espaços duplicados, espaços nas extremidades e converte para maiúsculas)

In [ ]:
def normalize_text(text):
    if pd.isna(text):
        return text
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    text = text.upper()
    text = "".join(c for c in unicodedata.normalize("NFKD", text) if not unicodedata.combining(c))
    return text

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].apply(normalize_text)

## 4. Missing values semânticos

In [ ]:
missing_values = {"","N/D","NULO","NULL","UNDEFINED","NAO INFORMADO","NAO INFORMACAO","SEM INFORMACAO","SEM INFOMACAO","NAO SABE","NAO SABE INFORMAR","DENUNCIANTE NAO SOUBE INFORMAR","ATENDIMENTO INTERROMPIDO","ATENDIMENTO INTERRPOMPIDO"}

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].replace(list(missing_values), np.nan)

## 5. Extrair código e nome dos municípios

In [ ]:
def extrair_codigo(v):
    if pd.isna(v): return pd.NA
    m = re.match(r"^(\d+)\s*\|", str(v))
    return m.group(1) if m else pd.NA

def extrair_nome(v):
    if pd.isna(v): return pd.NA
    v=str(v)
    return v.split("|")[-1].strip() if "|" in v else v.strip()

municipios=["municipio","municipio_da_vitima","sl_suspeito_municipio","sl_vitima_naturalizado_municipio","sl_suspeito_naturalizado_municipio"]
for c in municipios:
    if c in df.columns:
        df[f"{c}_codigo"] = df[c].apply(extrair_codigo)
        df[f"{c}_nome"] = df[c].apply(extrair_nome)

## 6. Padronização da relação vítima-suspeito

In [ ]:
if "relacao_vitima_suspeito" in df.columns:
    df["relacao_vitima_suspeito"]=(df["relacao_vitima_suspeito"]
      .str.replace(r"\s+\(","(",regex=True)
      .str.replace(r"\)\s+",")",regex=True))

## 7. Conversão de faixas etárias

In [ ]:
def parse_age_range(valor):
    if pd.isna(valor):
        return pd.Series([np.nan,np.nan])
    valor=str(valor)
    m=re.search(r"(\d+)\s*A\s*(\d+)",valor)
    if m: return pd.Series([int(m.group(1)),int(m.group(2))])
    m=re.search(r"^(\d+)\s*ANO",valor)
    if m:
        i=int(m.group(1)); return pd.Series([i,i])
    if "MENOS DE 01 ANO" in valor or "RECEM" in valor or "VIDA INTRAUTERINA" in valor:
        return pd.Series([0,0])
    if "90+" in valor:
        return pd.Series([90,120])
    return pd.Series([np.nan,np.nan])

if "faixa_etaria_da_vitima" in df.columns:
    df[["idade_min_vitima","idade_max_vitima"]]=df["faixa_etaria_da_vitima"].apply(parse_age_range)
    df["idade_media_vitima"]=(df["idade_min_vitima"]+df["idade_max_vitima"])/2

if "faixa_etaria_do_suspeito" in df.columns:
    df[["idade_min_suspeito","idade_max_suspeito"]]=df["faixa_etaria_do_suspeito"].apply(parse_age_range)
    df["idade_media_suspeito"]=(df["idade_min_suspeito"]+df["idade_max_suspeito"])/2

## 8. Hierarquias de violação e motivação

In [ ]:
def split_h(text, sep):
    if pd.isna(text): return pd.Series([None,None,None])
    p=[x.strip() for x in str(text).split(sep)]
    p += [None]*(3-len(p))
    return pd.Series(p[:3])

if "violacao" in df.columns:
    df[["violacao_n1","violacao_n2","violacao_n3"]]=df["violacao"].apply(lambda x: split_h(x, ">"))
if "motivacao" in df.columns:
    df[["motivacao_n1","motivacao_n2","motivacao_n3"]]=df["motivacao"].apply(lambda x: split_h(x, "."))

## 9. Perfil da vítima e do suspeito

In [ ]:
df["vitima_vulneravel"] = (((df.get("idade_media_vitima")<18) | (df.get("idade_media_vitima")>=60))).astype("Int64")

df["gap_etario"] = df.get("idade_media_suspeito") - df.get("idade_media_vitima")
df["suspeito_20_anos_mais_velho"]=(df["gap_etario"]>=20).astype("Int64")

## 10. Proximidade geográfica

In [ ]:
if {"uf_da_vitima","uf_do_suspeito"}.issubset(df.columns):
    df["mesma_uf"]=(df["uf_da_vitima"]==df["uf_do_suspeito"]).astype(int)

if {"pais_da_vitima","pais_do_suspeito"}.issubset(df.columns):
    df["mesmo_pais"]=(df["pais_da_vitima"]==df["pais_do_suspeito"]).astype(int)

## 11. Features temporais

In [ ]:
df["data_de_cadastro"] = pd.to_datetime(df["data_de_cadastro"], errors="coerce")
df["ano"] = df["data_de_cadastro"].dt.year
df["mes"] = df["data_de_cadastro"].dt.month
df["trimestre"] = df["data_de_cadastro"].dt.quarter
df["dia"] = df["data_de_cadastro"].dt.day
df["dia_semana"] = df["data_de_cadastro"].dt.dayofweek
df["fim_de_semana"]=(df["dia_semana"]>=5).astype(int)

## 12. Encoding baseado em frequência para alta cardinalidade

In [ ]:
cat_cols=df.select_dtypes(include=["object","category"]).columns
for col in cat_cols:
    if df[col].nunique(dropna=True)>50:
        freq=df[col].value_counts(normalize=True)
        df[f"{col}_freq"] = df[col].map(freq)

## 13. Relatório final de cardinalidade

In [ ]:
pd.DataFrame({"coluna":df.columns,"n_unique":[df[c].nunique(dropna=True) for c in df.columns]}).sort_values("n_unique",ascending=False).head(100)

## 14. Exportação

In [ ]:
df.to_parquet("disque100_feature_engineering.parquet")
df.to_csv("disque100_feature_engineering.csv", index=False)